# KL Perturb v2: OGBN-Arxiv (Runtime-Bounded)

This notebook is a refactor of the KL-perturbed Edge-DP prototype pipeline for larger graphs.

Design goals:
- Target `ogbn-arxiv` by default
- Keep runtime bounded with capped splits and capped prototype size
- Run a lightweight epsilon sweep instead of full ablations

You can still switch to Cora by changing `CONFIG['dataset_name']`.

In [1]:
import importlib.util
import subprocess
import sys

def ensure_package(module_name, pip_name=None):
    if importlib.util.find_spec(module_name) is None:
        pkg = pip_name or module_name
        print(f'Installing {pkg} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

ensure_package('torch_geometric', 'torch-geometric')
ensure_package('ogb', 'ogb')

import random
import time
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.datasets import Planetoid
from torch_geometric.utils import subgraph, k_hop_subgraph, to_undirected, coalesce, dropout_edge
from torch_geometric.nn import GCNConv, global_mean_pool

from ogb.nodeproppred import PygNodePropPredDataset

try:
    import pandas as pd
except Exception:
    pd = None

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Active device: {device}')

Installing torch-geometric ...
Installing ogb ...
Active device: cuda


In [2]:
import pickle
from contextlib import contextmanager


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _register_pyg_safe_globals():
    """Allowlist PyG classes for PyTorch 2.6+ weights-only unpickling."""
    if not hasattr(torch.serialization, 'add_safe_globals'):
        return

    safe = []
    try:
        from torch_geometric.data.data import Data
        safe.append(Data)
    except Exception:
        pass

    try:
        from torch_geometric.data.data import DataEdgeAttr
        safe.append(DataEdgeAttr)
    except Exception:
        pass

    try:
        from torch_geometric.data.data import DataTensorAttr
        safe.append(DataTensorAttr)
    except Exception:
        pass

    try:
        from torch_geometric.data.storage import BaseStorage, NodeStorage, EdgeStorage, GlobalStorage
        safe.extend([BaseStorage, NodeStorage, EdgeStorage, GlobalStorage])
    except Exception:
        pass

    if len(safe) > 0:
        torch.serialization.add_safe_globals(safe)


@contextmanager
def _torch_load_weights_only_false():
    """Temporary compatibility shim for trusted local dataset files."""
    original_torch_load = torch.load

    def patched_torch_load(*args, **kwargs):
        kwargs.setdefault('weights_only', False)
        return original_torch_load(*args, **kwargs)

    torch.load = patched_torch_load
    try:
        yield
    finally:
        torch.load = original_torch_load


def load_graph_dataset(dataset_name='ogbn-arxiv'):
    name = dataset_name.lower()

    if name == 'ogbn-arxiv':
        _register_pyg_safe_globals()
        try:
            dataset = PygNodePropPredDataset(name='ogbn-arxiv', root='data/ogb')
        except pickle.UnpicklingError:
            print('Encountered PyTorch 2.6 weights-only loading issue; retrying with weights_only=False for trusted OGB files...')
            with _torch_load_weights_only_false():
                dataset = PygNodePropPredDataset(name='ogbn-arxiv', root='data/ogb')

        data = dataset[0]
        data.y = data.y.squeeze(-1).long()
        data.x = data.x.float()
        num_classes = int(dataset.num_classes)
    elif name == 'cora':
        dataset = Planetoid(root='data/Planetoid', name='Cora')
        data = dataset[0]
        data.y = data.y.long()
        data.x = data.x.float()
        num_classes = int(dataset.num_classes)
    else:
        raise ValueError(f'Unsupported dataset: {dataset_name}')

    data.edge_index = to_undirected(data.edge_index, num_nodes=data.num_nodes)
    data.edge_index = coalesce(data.edge_index, num_nodes=data.num_nodes)
    data.x = F.normalize(data.x, p=2, dim=1)

    return dataset, data, num_classes


def stratified_split_indices(y, public_frac=0.02, val_frac=0.02, seed=0):
    g = torch.Generator().manual_seed(seed)
    classes = torch.unique(y)
    pub, val, train = [], [], []

    for c in classes.tolist():
        idx = torch.where(y == c)[0]
        if idx.numel() == 0:
            continue
        perm = idx[torch.randperm(idx.numel(), generator=g)]

        n_pub = max(1, int(public_frac * idx.numel()))
        n_val = max(1, int(val_frac * idx.numel()))
        n_pub = min(n_pub, idx.numel())
        n_val = min(n_val, max(idx.numel() - n_pub, 0))

        pub.append(perm[:n_pub])
        val.append(perm[n_pub:n_pub + n_val])
        train.append(perm[n_pub + n_val:])

    public_nodes = torch.cat(pub) if len(pub) > 0 else torch.tensor([], dtype=torch.long)
    val_nodes = torch.cat(val) if len(val) > 0 else torch.tensor([], dtype=torch.long)
    train_nodes = torch.cat(train) if len(train) > 0 else torch.tensor([], dtype=torch.long)

    public_nodes = public_nodes[torch.randperm(public_nodes.numel(), generator=g)]
    val_nodes = val_nodes[torch.randperm(val_nodes.numel(), generator=g)]
    train_nodes = train_nodes[torch.randperm(train_nodes.numel(), generator=g)]

    return public_nodes, train_nodes, val_nodes


def cap_indices(indices, max_count=None, seed=0):
    if max_count is None or indices.numel() <= max_count:
        return indices
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(indices.numel(), generator=g)
    return indices[perm[:max_count]]


def split_edges_by_nodes(edge_index, public_nodes, train_nodes, val_nodes, num_nodes):
    pub_edge_index, _ = subgraph(public_nodes, edge_index, relabel_nodes=False, num_nodes=num_nodes)
    private_edge_index, _ = subgraph(train_nodes, edge_index, relabel_nodes=False, num_nodes=num_nodes)
    val_edge_index, _ = subgraph(val_nodes, edge_index, relabel_nodes=False, num_nodes=num_nodes)
    return pub_edge_index, private_edge_index, val_edge_index

In [3]:
from collections import Counter


def one_hop_mean_aggregate(edge_index, x_features, num_nodes):
    edge_undirected = to_undirected(edge_index, num_nodes=num_nodes)
    edge_undirected = coalesce(edge_undirected, num_nodes=num_nodes)

    H_sum = torch.zeros_like(x_features)
    if edge_undirected.numel() == 0:
        return H_sum

    row, col = edge_undirected
    H_sum.scatter_add_(0, row.unsqueeze(1).expand(-1, x_features.size(1)), x_features[col])

    counts = torch.zeros((num_nodes,), dtype=x_features.dtype, device=x_features.device)
    counts.scatter_add_(0, row, torch.ones_like(row, dtype=x_features.dtype))

    H_mean = torch.zeros_like(H_sum)
    nonzero = counts > 0
    H_mean[nonzero] = H_sum[nonzero] / counts[nonzero].unsqueeze(1)
    return H_mean


def degree_weighted_sgc_embedding(x_sub, edge_index_sub):
    n = x_sub.size(0)
    if n == 0:
        return torch.zeros(x_sub.size(1), device=x_sub.device)

    if edge_index_sub.numel() == 0:
        weights = torch.ones(n, dtype=x_sub.dtype, device=x_sub.device)
    else:
        row, col = edge_index_sub
        deg = torch.bincount(row, minlength=n).to(x_sub.dtype)
        deg = deg + torch.bincount(col, minlength=n).to(x_sub.dtype)
        weights = deg + 1.0

    weighted_sum = (x_sub * weights.unsqueeze(1)).sum(dim=0)
    return weighted_sum / weights.sum().clamp_min(1e-12)


def build_prototype_feature(edge_index_sub, x_sub, query_mode='one_hop'):
    num_sub_nodes = x_sub.size(0)
    H1_nodes = one_hop_mean_aggregate(edge_index_sub, x_sub, num_nodes=num_sub_nodes)
    z1 = degree_weighted_sgc_embedding(H1_nodes, edge_index_sub)

    if query_mode == 'one_hop':
        return F.normalize(z1, p=2, dim=0)

    if query_mode == 'two_hop_concat':
        H2_nodes = one_hop_mean_aggregate(edge_index_sub, H1_nodes, num_nodes=num_sub_nodes)
        z2 = degree_weighted_sgc_embedding(H2_nodes, edge_index_sub)
        return F.normalize(torch.cat([z1, z2], dim=0), p=2, dim=0)

    raise ValueError(f'Unsupported query_mode: {query_mode}')


def apply_query_normalization(Q, mode='none', clip_norm=1.0):
    if mode is None or mode == 'none':
        return Q

    if mode == 'l2_row':
        return F.normalize(Q, p=2, dim=1)

    if mode == 'clipped_l2':
        norms = torch.linalg.norm(Q, dim=1, keepdim=True).clamp_min(1e-12)
        scale = torch.clamp(float(clip_norm) / norms, max=1.0)
        return Q * scale

    raise ValueError(f'Unsupported query_normalization: {mode}')


def build_private_query_features(
    private_edges,
    x_features,
    query_mode='one_hop',
    query_normalization='none',
    query_clip_norm=1.0,
):
    num_nodes = x_features.size(0)
    H1 = one_hop_mean_aggregate(private_edges, x_features, num_nodes=num_nodes)

    if query_mode == 'one_hop':
        Q = H1
    elif query_mode == 'two_hop_concat':
        H2 = one_hop_mean_aggregate(private_edges, H1, num_nodes=num_nodes)
        Q = torch.cat([H1, H2], dim=1)
    else:
        raise ValueError(f'Unsupported query_mode: {query_mode}')

    return apply_query_normalization(Q, mode=query_normalization, clip_norm=query_clip_norm)


def build_class_conditional_public_dictionary(
    data_obj,
    x_features,
    labels_tensor,
    public_nodes_tensor,
    pub_edges,
    num_classes,
    dict_per_class=12,
    num_hops=1,
    query_mode='one_hop',
    min_class_fraction=1.0,
    max_proto_nodes=64,
    return_diagnostics=False,
    rng_seed=None,
):
    print(
        f'Building dictionary ({dict_per_class}/class, hops={num_hops}, mode={query_mode}, max_proto_nodes={max_proto_nodes})'
    )

    pub_undirected = to_undirected(pub_edges, num_nodes=data_obj.num_nodes)
    pub_undirected = coalesce(pub_undirected, num_nodes=data_obj.num_nodes)

    generator = None
    if rng_seed is not None:
        generator = torch.Generator().manual_seed(int(rng_seed))

    public_labels = labels_tensor[public_nodes_tensor]
    dictionary = []
    class_to_proto_indices = {c: [] for c in range(num_classes)}

    proto_node_counts = []
    proto_edge_counts = []

    for c in range(num_classes):
        class_pool = public_nodes_tensor[public_labels == c]
        if class_pool.numel() == 0:
            continue

        for _ in range(dict_per_class):
            if generator is None:
                anchor = class_pool[torch.randint(0, class_pool.numel(), (1,)).item()]
            else:
                anchor = class_pool[torch.randint(0, class_pool.numel(), (1,), generator=generator).item()]

            subset, _, _, _ = k_hop_subgraph(
                int(anchor.item()),
                num_hops=num_hops,
                edge_index=pub_undirected,
                relabel_nodes=False,
                num_nodes=data_obj.num_nodes,
            )

            proto_labels = labels_tensor[subset]
            class_mask = proto_labels == c
            class_count = int(class_mask.sum().item())
            subset_count = int(subset.numel())
            frac = class_count / max(subset_count, 1)
            if frac >= min_class_fraction and class_count > 0:
                class_subset = subset
            else:
                class_subset = subset[class_mask]
                if class_subset.numel() == 0:
                    class_subset = anchor.view(1)

            if class_subset.numel() > max_proto_nodes:
                if generator is None:
                    perm = torch.randperm(class_subset.numel())[:max_proto_nodes]
                else:
                    perm = torch.randperm(class_subset.numel(), generator=generator)[:max_proto_nodes]
                class_subset = class_subset[perm]

            sub_edge_index, _ = subgraph(
                class_subset,
                pub_undirected,
                relabel_nodes=True,
                num_nodes=data_obj.num_nodes,
            )

            x_sub = x_features[class_subset]
            proto_feat = build_prototype_feature(sub_edge_index, x_sub, query_mode=query_mode)

            proto_idx = len(dictionary)
            dictionary.append({
                'edge_index': sub_edge_index,
                'x': x_sub,
                'proto_feat': proto_feat,
                'class': c,
            })
            class_to_proto_indices[c].append(proto_idx)

            proto_node_counts.append(int(class_subset.numel()))
            proto_edge_counts.append(int(sub_edge_index.size(1)))

    if len(dictionary) == 0:
        raise RuntimeError('Dictionary is empty. Increase public split or reduce filtering constraints.')

    dict_feats = torch.stack([g['proto_feat'] for g in dictionary], dim=0)
    print(f'Dictionary size={len(dictionary)} | Feature dim={dict_feats.size(1)}')

    if not return_diagnostics:
        return dictionary, dict_feats, class_to_proto_indices

    per_class_counts = {int(c): int(len(class_to_proto_indices[c])) for c in range(num_classes)}
    node_hist = dict(Counter(proto_node_counts))

    singleton_fraction = float(sum(1 for n in proto_node_counts if n == 1) / max(len(proto_node_counts), 1))
    edgeless_fraction = float(sum(1 for e in proto_edge_counts if e == 0) / max(len(proto_edge_counts), 1))

    diagnostics = {
        'dict_size': int(len(dictionary)),
        'feature_dim': int(dict_feats.size(1)),
        'prototypes_per_class': per_class_counts,
        'prototype_node_count_hist': {int(k): int(v) for k, v in node_hist.items()},
        'prototype_node_count_mean': float(sum(proto_node_counts) / max(len(proto_node_counts), 1)),
        'prototype_node_count_max': int(max(proto_node_counts) if len(proto_node_counts) > 0 else 0),
        'singleton_fraction': singleton_fraction,
        'edgeless_fraction': edgeless_fraction,
        'prototype_edge_count_mean': float(sum(proto_edge_counts) / max(len(proto_edge_counts), 1)),
    }

    return dictionary, dict_feats, class_to_proto_indices, diagnostics


def gumbel_max_sample(logits):
    u = torch.rand_like(logits).clamp_(1e-8, 1 - 1e-8)
    gumbel = -torch.log(-torch.log(u))
    return torch.argmax(logits + gumbel).item()


@torch.no_grad()
def analyze_dictionary_query_fit(
    private_edges,
    x_features,
    labels,
    private_nodes_tensor,
    dict_features,
    class_to_proto_indices,
    query_mode='one_hop',
    query_normalization='none',
    query_clip_norm=1.0,
    max_nodes_per_class=2000,
):
    Q = build_private_query_features(
        private_edges,
        x_features,
        query_mode=query_mode,
        query_normalization=query_normalization,
        query_clip_norm=query_clip_norm,
    )

    num_classes = int(labels.max().item()) + 1
    per_class = {}
    all_distances = []

    private_labels = labels[private_nodes_tensor].long()
    for c in range(num_classes):
        class_nodes = private_nodes_tensor[private_labels == c]
        if class_nodes.numel() == 0:
            per_class[c] = None
            continue

        if class_nodes.numel() > max_nodes_per_class:
            keep = torch.randperm(class_nodes.numel())[:max_nodes_per_class]
            class_nodes = class_nodes[keep]

        proto_idx = class_to_proto_indices.get(c, [])
        if len(proto_idx) == 0:
            per_class[c] = None
            continue

        class_proto = dict_features[torch.tensor(proto_idx, dtype=torch.long)]
        class_query = Q[class_nodes]

        dists = torch.cdist(class_query, class_proto, p=2)
        nearest = dists.min(dim=1).values

        mean_d = float(nearest.mean().item())
        per_class[c] = mean_d
        all_distances.append(nearest)

    if len(all_distances) > 0:
        overall = float(torch.cat(all_distances).mean().item())
        used_nodes = int(torch.cat([t for t in all_distances]).numel())
    else:
        overall = float('nan')
        used_nodes = 0

    return {
        'overall_mean_within_class_nearest_distance': overall,
        'per_class_mean_nearest_distance': per_class,
        'used_private_nodes_for_distance': int(used_nodes),
    }


@torch.no_grad()
def synthesize_edge_dp_assignments(
    private_edges,
    x_features,
    labels,
    private_nodes_tensor,
    dict_features,
    class_to_proto_indices,
    epsilon_total,
    utility_sensitivity=1.0,
    query_mode='one_hop',
    label_conditioning=True,
    query_normalization='none',
    query_clip_norm=1.0,
    return_full_diagnostics=False,
):
    epsilon_per_node = epsilon_total / 2.0

    Q = build_private_query_features(
        private_edges,
        x_features,
        query_mode=query_mode,
        query_normalization=query_normalization,
        query_clip_norm=query_clip_norm,
    )
    K_global = dict_features.size(0)
    num_classes = int(labels.max().item()) + 1

    all_idx = torch.arange(K_global, dtype=torch.long)
    class_idx_tensors = {}
    for c in range(num_classes):
        idx_list = class_to_proto_indices.get(c, [])
        class_idx_tensors[c] = torch.tensor(idx_list, dtype=torch.long) if len(idx_list) > 0 else all_idx

    N = int(private_nodes_tensor.numel())
    selected_proto_indices = torch.empty(N, dtype=torch.long)
    private_labels = labels[private_nodes_tensor].long().clone()

    selection_counts = torch.zeros(K_global, dtype=torch.long)

    entropy_sum = 0.0
    top_prob_sum = 0.0
    nearest_dist_sum = 0.0
    sampled_dist_sum = 0.0

    entropies = [] if return_full_diagnostics else None
    max_probs = [] if return_full_diagnostics else None
    nearest_distances = [] if return_full_diagnostics else None
    sampled_distances = [] if return_full_diagnostics else None

    for j, u in enumerate(private_nodes_tensor.tolist()):
        y_u = int(labels[u].item())
        candidate_idx = class_idx_tensors[y_u] if label_conditioning else all_idx

        candidate_feats = dict_features.index_select(0, candidate_idx)
        q_u = Q[u]

        distances = torch.linalg.norm(candidate_feats - q_u.unsqueeze(0), dim=1)
        scores = -distances
        logits = (epsilon_per_node / (2.0 * utility_sensitivity)) * scores
        probs = torch.softmax(logits, dim=0)

        entropy = -(probs * torch.log(probs.clamp_min(1e-12))).sum().item()
        top_prob = float(probs.max().item())

        local_idx = gumbel_max_sample(logits)
        proto_idx = int(candidate_idx[local_idx].item())

        nearest_dist = float(distances.min().item())
        sampled_dist = float(distances[local_idx].item())

        entropy_sum += entropy
        top_prob_sum += top_prob
        nearest_dist_sum += nearest_dist
        sampled_dist_sum += sampled_dist

        if return_full_diagnostics:
            entropies.append(entropy)
            max_probs.append(top_prob)
            nearest_distances.append(nearest_dist)
            sampled_distances.append(sampled_dist)

        selected_proto_indices[j] = proto_idx
        selection_counts[proto_idx] += 1

        if (j + 1) % 5000 == 0:
            print(f'  sampled {j + 1}/{N} private nodes')

    used_ratio = float((selection_counts > 0).sum().item()) / float(max(K_global, 1))
    unused_fraction = float((selection_counts == 0).sum().item()) / float(max(K_global, 1))

    class_selected_counts = {}
    covered_class_count = 0
    for c in range(num_classes):
        idx = class_to_proto_indices.get(c, [])
        if len(idx) == 0:
            class_selected_counts[c] = 0
            continue
        count = int(selection_counts[torch.tensor(idx, dtype=torch.long)].sum().item())
        class_selected_counts[c] = count
        if count > 0:
            covered_class_count += 1

    diagnostics = {
        'epsilon_per_node': float(epsilon_per_node),
        'dict_size': int(K_global),
        'mean_entropy': float(entropy_sum / max(N, 1)),
        'mean_top_probability': float(top_prob_sum / max(N, 1)),
        'mean_nearest_distance': float(nearest_dist_sum / max(N, 1)),
        'mean_sampled_distance': float(sampled_dist_sum / max(N, 1)),
        'unique_selected_ratio': used_ratio,
        'unused_prototype_fraction': unused_fraction,
        'class_coverage_selected_prototypes': float(covered_class_count / max(num_classes, 1)),
        'class_selected_assignment_counts': {int(k): int(v) for k, v in class_selected_counts.items()},
        'selection_frequency_per_prototype': selection_counts.tolist(),
    }

    if return_full_diagnostics:
        entropy_tensor = torch.tensor(entropies, dtype=torch.float32)
        prob_tensor = torch.tensor(max_probs, dtype=torch.float32)
        nearest_tensor = torch.tensor(nearest_distances, dtype=torch.float32)
        sampled_tensor = torch.tensor(sampled_distances, dtype=torch.float32)

        diagnostics.update({
            'entropy_values': entropies,
            'top_probability_values': max_probs,
            'nearest_distance_values': nearest_distances,
            'sampled_distance_values': sampled_distances,
            'entropy_hist_20': torch.histc(entropy_tensor, bins=20).tolist() if entropy_tensor.numel() > 0 else [],
            'top_probability_hist_20': torch.histc(prob_tensor, bins=20, min=0.0, max=1.0).tolist() if prob_tensor.numel() > 0 else [],
            'nearest_distance_hist_20': torch.histc(nearest_tensor, bins=20).tolist() if nearest_tensor.numel() > 0 else [],
            'sampled_distance_hist_20': torch.histc(sampled_tensor, bins=20).tolist() if sampled_tensor.numel() > 0 else [],
        })

    assignments = {'proto_indices': selected_proto_indices, 'labels': private_labels}
    return assignments, diagnostics


class PrototypeAssignmentDataset(torch.utils.data.Dataset):
    def __init__(self, public_dict, proto_indices, labels):
        self.public_dict = public_dict
        self.proto_indices = proto_indices.long().cpu()
        self.labels = labels.long().cpu()

    def __len__(self):
        return self.proto_indices.numel()

    def __getitem__(self, idx):
        proto = self.public_dict[int(self.proto_indices[idx].item())]
        return Data(x=proto['x'], edge_index=proto['edge_index'], y=self.labels[idx].view(1))


class StandardGCN(nn.Module):
    def __init__(self, hidden_channels, num_features, num_classes):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.lin = nn.Linear(hidden_channels, num_classes)

    def forward(self, x, edge_index, batch):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index).relu()
        x = global_mean_pool(x, batch)
        return self.lin(x)


def build_validation_loader(x_l2, labels, val_nodes, val_edge_index, query_hops=1, max_val_graphs=3000, seed=0):
    val_nodes = cap_indices(val_nodes, max_count=max_val_graphs, seed=seed)
    val_undirected = to_undirected(val_edge_index, num_nodes=x_l2.size(0))
    val_undirected = coalesce(val_undirected, num_nodes=x_l2.size(0))

    data_list = []
    for node in val_nodes.tolist():
        subset, sub_edge_index, _, _ = k_hop_subgraph(
            node,
            num_hops=query_hops,
            edge_index=val_undirected,
            relabel_nodes=True,
            num_nodes=x_l2.size(0),
        )
        data_list.append(Data(x=x_l2[subset], edge_index=sub_edge_index, y=labels[node].unsqueeze(0)))

    return DataLoader(data_list, batch_size=128, shuffle=False), val_nodes


def train_gnn_from_assignments(
    assignments,
    public_dict,
    val_loader,
    num_features,
    num_classes,
    epochs=12,
    batch_size=64,
    lr=0.01,
    feature_jitter_std=0.03,
    edge_dropout_p=0.10,
):
    train_dataset = PrototypeAssignmentDataset(public_dict, assignments['proto_indices'], assignments['labels'])
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    model = StandardGCN(hidden_channels=32, num_features=num_features, num_classes=num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    use_cuda_amp = device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_cuda_amp)

    best_val = 0.0
    for epoch in range(1, epochs + 1):
        model.train()
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad(set_to_none=True)

            x_aug = batch.x + torch.randn_like(batch.x) * feature_jitter_std
            edge_index_aug, _ = dropout_edge(batch.edge_index, p=edge_dropout_p)
            if edge_index_aug.numel() == 0:
                edge_index_aug = batch.edge_index

            if use_cuda_amp:
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    out = model(x_aug, edge_index_aug, batch.batch)
                    loss = criterion(out, batch.y)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                out = model(x_aug, edge_index_aug, batch.batch)
                loss = criterion(out, batch.y)
                loss.backward()
                optimizer.step()

        model.eval()
        val_correct = 0
        with torch.no_grad():
            for vb in val_loader:
                vb = vb.to(device)
                pred = model(vb.x, vb.edge_index, vb.batch).argmax(dim=1)
                val_correct += int((pred == vb.y).sum())

        val_acc = val_correct / len(val_loader.dataset)
        best_val = max(best_val, val_acc)

        if epoch == 1 or epoch % 4 == 0:
            print(f'Epoch {epoch:03d} | Val Acc: {val_acc:.4f} | Best: {best_val:.4f}')

    return best_val

In [ ]:
from typing import Any, Dict, List, Tuple


def get_nesterov_sgd_ablation_config() -> Dict[str, Any]:
    return {
        'dataset_name': 'ogbn-arxiv',
        'seed': 42,
        'public_frac': 0.02,
        'val_frac': 0.02,
        'max_private_nodes': 30000,
        'max_val_nodes': 3000,
        'dict_per_class': 48,
        'walk_hops': 1,
        'query_hops': 2,
        'query_mode': 'one_hop',
        'label_conditioning': True,
        'min_class_fraction': 1.0,
        'max_proto_nodes': 64,
        'epochs': 100,  # Increased for SGD convergence
        'batch_size': 64,
        'lr': 0.1,      # Higher LR for SGD
        'weight_decay': 1e-4,
        'feature_jitter_std': 0.01,
        'edge_dropout_p': 0.03,
        'epsilon_list': [0.5, 1.0, 4.0, 100.0],
        'patience': 15  # Relaxed patience for bumpy SGD curves
    }


def prepare_nesterov_sgd_ablation_state(config: Dict[str, Any]) -> Dict[str, Any]:
    set_seed(config['seed'])
    dataset, data, num_classes = load_graph_dataset(config['dataset_name'])
    x_l2 = data.x

    public_nodes, train_nodes, val_nodes = stratified_split_indices(
        data.y,
        public_frac=config['public_frac'],
        val_frac=config['val_frac'],
        seed=config['seed'],
    )

    train_nodes = cap_indices(train_nodes, config['max_private_nodes'], seed=config['seed'] + 1)
    val_nodes = cap_indices(val_nodes, config['max_val_nodes'], seed=config['seed'] + 2)

    print(f"Dataset={config['dataset_name']} | Nodes={data.num_nodes} | Edges={data.edge_index.size(1)}")

    pub_edge_index, private_edge_index, val_edge_index = split_edges_by_nodes(
        data.edge_index, public_nodes, train_nodes, val_nodes, data.num_nodes
    )

    public_dict, public_proto_feats, class_to_proto_indices = build_class_conditional_public_dictionary(
        data_obj=data,
        x_features=x_l2,
        labels_tensor=data.y,
        public_nodes_tensor=public_nodes,
        pub_edges=pub_edge_index,
        num_classes=num_classes,
        dict_per_class=config['dict_per_class'],
        num_hops=config['walk_hops'],
        query_mode=config['query_mode'],
        min_class_fraction=config['min_class_fraction'],
        max_proto_nodes=config['max_proto_nodes'],
    )

    val_loader, val_nodes_used = build_validation_loader(
        x_l2=x_l2,
        labels=data.y,
        val_nodes=val_nodes,
        val_edge_index=val_edge_index,
        query_hops=config['query_hops'],
        max_val_graphs=config['max_val_nodes'],
        seed=config['seed'] + 3,
    )

    utility_sensitivity = 2.0 if config['query_mode'] == 'two_hop_concat' else 1.0

    return {
        'dataset': dataset,
        'data': data,
        'num_classes': num_classes,
        'x_l2': x_l2,
        'public_nodes': public_nodes,
        'train_nodes': train_nodes,
        'val_nodes': val_nodes,
        'private_edge_index': private_edge_index,
        'public_dict': public_dict,
        'public_proto_feats': public_proto_feats,
        'class_to_proto_indices': class_to_proto_indices,
        'val_loader': val_loader,
        'val_nodes_used': val_nodes_used,
        'utility_sensitivity': utility_sensitivity,
    }


def run_single_nesterov_sgd_trial(
    epsilon: float,
    config: Dict[str, Any],
    state: Dict[str, Any],
) -> Dict[str, float]:
    print('\n' + '=' * 60)
    print(f'Running epsilon={epsilon} with Nesterov SGD ({config["epochs"]} Epochs)')

    t0 = time.time()
    assignments, _ = synthesize_edge_dp_assignments(
        private_edges=state['private_edge_index'],
        x_features=state['x_l2'],
        labels=state['data'].y,
        private_nodes_tensor=state['train_nodes'],
        dict_features=state['public_proto_feats'],
        class_to_proto_indices=state['class_to_proto_indices'],
        epsilon_total=epsilon,
        utility_sensitivity=state['utility_sensitivity'],
        query_mode=config['query_mode'],
        label_conditioning=config['label_conditioning'],
    )

    best_val_acc = train_gnn_from_assignments_optimized(
        assignments=assignments,
        public_dict=state['public_dict'],
        val_loader=state['val_loader'],
        num_features=state['data'].x.size(1),
        num_classes=state['num_classes'],
        epochs=config['epochs'],
        batch_size=config['batch_size'],
        lr=config['lr'],
        weight_decay=config['weight_decay'],
        feature_jitter_std=config['feature_jitter_std'],
        edge_dropout_p=config['edge_dropout_p'],
        patience=config['patience'],
    )

    elapsed = time.time() - t0
    print(f"Result for eps={epsilon}: best_val_acc={best_val_acc:.4f} | time={elapsed:.1f}s")

    return {
        'epsilon_total': float(epsilon),
        'best_val_acc': float(best_val_acc),
        'elapsed_sec': float(elapsed),
    }


def run_nesterov_sgd_ablation(config: Dict[str, Any]) -> Tuple[List[Dict[str, float]], Dict[str, Any]]:
    state = prepare_nesterov_sgd_ablation_state(config)
    results = [
        run_single_nesterov_sgd_trial(float(eps), config, state)
        for eps in config['epsilon_list']
    ]
    return results, state


CONFIG = get_nesterov_sgd_ablation_config()
results, ABLATION_STATE = run_nesterov_sgd_ablation(CONFIG)

# Keep prior top-level names available for downstream cells.
dataset = ABLATION_STATE['dataset']
data = ABLATION_STATE['data']
num_classes = ABLATION_STATE['num_classes']
x_l2 = ABLATION_STATE['x_l2']
public_nodes = ABLATION_STATE['public_nodes']
train_nodes = ABLATION_STATE['train_nodes']
val_nodes = ABLATION_STATE['val_nodes']
private_edge_index = ABLATION_STATE['private_edge_index']
public_dict = ABLATION_STATE['public_dict']
public_proto_feats = ABLATION_STATE['public_proto_feats']
class_to_proto_indices = ABLATION_STATE['class_to_proto_indices']
val_loader = ABLATION_STATE['val_loader']
val_nodes_used = ABLATION_STATE['val_nodes_used']
utility_sensitivity = ABLATION_STATE['utility_sensitivity']

if pd is not None:
    display(pd.DataFrame(results))

Dataset=ogbn-arxiv | Nodes=169343 | Edges=2315598
Building dictionary (48/class, hops=1, mode=one_hop, max_proto_nodes=64)
Dictionary size=1920 | Feature dim=128

Running epsilon=0.5 with Nesterov SGD (100 Epochs)
  sampled 5000/30000 private nodes
  sampled 10000/30000 private nodes
  sampled 15000/30000 private nodes
  sampled 20000/30000 private nodes
  sampled 25000/30000 private nodes
  sampled 30000/30000 private nodes


NameError: name 'make_fast_assignment_loader' is not defined

In [ ]:
# ==========================================
# CELL 6: STRONGER FAIRNESS BENCHMARK (MATCHED MODEL + MULTI-SEED)
# ==========================================

required_vars = ['data', 'x_l2', 'train_nodes', 'val_nodes', 'private_edge_index', 'val_edge_index', 'CONFIG', 'num_classes']
missing = [v for v in required_vars if v not in globals()]
if len(missing) > 0:
    raise RuntimeError(f"Missing required variables from earlier cells: {missing}")


def build_node_subgraph_loader(
    nodes,
    edge_index,
    x_features,
    labels,
    query_hops=1,
    max_graphs=6000,
    batch_size=64,
    shuffle=True,
    seed=0,
):
    used_nodes = cap_indices(nodes, max_count=max_graphs, seed=seed)
    edge_undirected = to_undirected(edge_index, num_nodes=x_features.size(0))
    edge_undirected = coalesce(edge_undirected, num_nodes=x_features.size(0))

    data_list = []
    for node in used_nodes.tolist():
        subset, sub_edge_index, _, _ = k_hop_subgraph(
            node,
            num_hops=query_hops,
            edge_index=edge_undirected,
            relabel_nodes=True,
            num_nodes=x_features.size(0),
        )
        data_list.append(Data(x=x_features[subset], edge_index=sub_edge_index, y=labels[node].unsqueeze(0)))

    if len(data_list) == 0:
        raise RuntimeError('No graphs were built for benchmark loader.')

    return DataLoader(data_list, batch_size=batch_size, shuffle=shuffle), used_nodes


def train_gcn_with_loaders(
    train_loader,
    val_loader,
    num_features,
    num_classes,
    epochs=12,
    lr=0.01,
    feature_jitter_std=0.03,
    edge_dropout_p=0.10,
):
    model = StandardGCN(hidden_channels=32, num_features=num_features, num_classes=num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    use_cuda_amp = device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_cuda_amp)

    best_val = 0.0
    for epoch in range(1, epochs + 1):
        model.train()
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad(set_to_none=True)

            x_aug = batch.x + torch.randn_like(batch.x) * feature_jitter_std
            edge_index_aug, _ = dropout_edge(batch.edge_index, p=edge_dropout_p)
            if edge_index_aug.numel() == 0:
                edge_index_aug = batch.edge_index

            if use_cuda_amp:
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    out = model(x_aug, edge_index_aug, batch.batch)
                    loss = criterion(out, batch.y)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                out = model(x_aug, edge_index_aug, batch.batch)
                loss = criterion(out, batch.y)
                loss.backward()
                optimizer.step()

        model.eval()
        val_correct = 0
        with torch.no_grad():
            for vb in val_loader:
                vb = vb.to(device)
                pred = model(vb.x, vb.edge_index, vb.batch).argmax(dim=1)
                val_correct += int((pred == vb.y).sum())

        val_acc = val_correct / len(val_loader.dataset)
        best_val = max(best_val, val_acc)

        if epoch == 1 or epoch % 4 == 0:
            print(f"  GCN epoch {epoch:02d} | val_acc={val_acc:.4f} | best={best_val:.4f}")

    return best_val


def clip_rows_l2(x, max_norm=1.0):
    norms = torch.linalg.norm(x, dim=1, keepdim=True).clamp_min(1e-12)
    scale = torch.clamp(max_norm / norms, max=1.0)
    return x * scale


def calibrate_gaussian_sensitivity(feature_dim, clip_norm=1.0, mode='moderate', sensitivity_override=None):
    if sensitivity_override is not None:
        return float(sensitivity_override)

    if mode == 'conservative':
        # Very pessimistic global bound.
        return 2.0 * float(clip_norm)
    if mode == 'moderate':
        # Practical per-dimension calibration for high-dimensional clipped vectors.
        return float(clip_norm) / math.sqrt(max(feature_dim, 1))
    if mode == 'dim_scaled':
        return (2.0 * float(clip_norm)) / math.sqrt(max(feature_dim, 1))

    raise ValueError(f'Unknown sensitivity mode: {mode}')


@torch.no_grad()
def edge_dp_noisy_sgc_features(
    private_edges,
    val_edges,
    x_features,
    num_nodes,
    epsilon=1.0,
    delta=1e-5,
    clip_norm=1.0,
    sensitivity_mode='moderate',
    sensitivity_override=None,
):
    """
    Continuous edge-DP feature baseline with clipped/calibrated Gaussian mechanism.
    """
    H_private = one_hop_mean_aggregate(private_edges, x_features, num_nodes=num_nodes)
    H_val = one_hop_mean_aggregate(val_edges, x_features, num_nodes=num_nodes)

    H_private = clip_rows_l2(H_private, max_norm=clip_norm)
    H_val = clip_rows_l2(H_val, max_norm=clip_norm)

    delta_l2 = calibrate_gaussian_sensitivity(
        feature_dim=H_private.size(1),
        clip_norm=clip_norm,
        mode=sensitivity_mode,
        sensitivity_override=sensitivity_override,
    )
    sigma = (delta_l2 / epsilon) * math.sqrt(2.0 * math.log(1.25 / delta))

    noise = torch.randn_like(H_private) * sigma
    H_private_noisy = H_private + noise

    signal_rms = float(H_private.pow(2).mean().sqrt().item())
    noise_rms = float(noise.pow(2).mean().sqrt().item())
    snr = signal_rms / max(noise_rms, 1e-12)

    diagnostics = {
        'gaussian_sensitivity': float(delta_l2),
        'gaussian_sigma': float(sigma),
        'signal_rms': float(signal_rms),
        'noise_rms': float(noise_rms),
        'snr': float(snr),
    }

    if snr < 0.15:
        print(f"  [warn] Noisy-SGC SNR is very low ({snr:.3f}); expect collapse at this epsilon/calibration.")

    return H_private_noisy, H_val, diagnostics


@torch.no_grad()
def edge_rr_perturb_edges(private_edges, train_nodes_tensor, num_nodes, epsilon=1.0):
    """
    Randomized-response style edge perturbation baseline.
    Runtime-bounded approximation suitable for OGBN-Arxiv sweeps.
    """
    p_keep = float(math.exp(epsilon) / (math.exp(epsilon) + 1.0))

    kept_edges, _ = dropout_edge(private_edges, p=(1.0 - p_keep))
    m = int(private_edges.size(1))
    add_count = int(((1.0 - p_keep) / max(p_keep, 1e-8)) * m)

    if add_count > 0:
        src = train_nodes_tensor[torch.randint(0, train_nodes_tensor.numel(), (add_count,))]
        dst = train_nodes_tensor[torch.randint(0, train_nodes_tensor.numel(), (add_count,))]
        add_edges = torch.stack([src, dst], dim=0)
        rr_edges = torch.cat([kept_edges, add_edges], dim=1)
    else:
        rr_edges = kept_edges

    rr_edges = to_undirected(rr_edges, num_nodes=num_nodes)
    rr_edges = coalesce(rr_edges, num_nodes=num_nodes)
    return rr_edges, p_keep


def run_fair_edge_dp_baselines_one_seed(
    epsilon=1.0,
    delta=1e-5,
    seed=0,
    train_cap=6000,
    epochs=12,
    gaussian_clip_norm=1.0,
    gaussian_sensitivity_mode='moderate',
    gaussian_sensitivity_override=None,
):
    set_seed(seed)
    print(f"\n--- Fair benchmark | seed={seed} | epsilon={epsilon} ---")

    # Shared validation loader (clean graph/features) for fair side-by-side model comparison.
    clean_val_loader, val_nodes_used = build_node_subgraph_loader(
        nodes=val_nodes,
        edge_index=val_edge_index,
        x_features=x_l2,
        labels=data.y,
        query_hops=CONFIG['query_hops'],
        max_graphs=CONFIG['max_val_nodes'],
        batch_size=CONFIG['batch_size'],
        shuffle=False,
        seed=seed + 101,
    )

    # Upper bound: non-private training on clean private graph.
    clean_train_loader, train_nodes_used = build_node_subgraph_loader(
        nodes=train_nodes,
        edge_index=private_edge_index,
        x_features=x_l2,
        labels=data.y,
        query_hops=CONFIG['query_hops'],
        max_graphs=train_cap,
        batch_size=CONFIG['batch_size'],
        shuffle=True,
        seed=seed + 102,
    )
    acc_clean_gcn = train_gcn_with_loaders(
        train_loader=clean_train_loader,
        val_loader=clean_val_loader,
        num_features=x_l2.size(1),
        num_classes=num_classes,
        epochs=epochs,
        feature_jitter_std=CONFIG['feature_jitter_std'],
        edge_dropout_p=CONFIG['edge_dropout_p'],
    )

    # Baseline A: Gaussian mechanism on 1-hop SGC-like node features, same downstream GCN.
    H_train_noisy, H_val_sgc, noisy_diag = edge_dp_noisy_sgc_features(
        private_edges=private_edge_index,
        val_edges=val_edge_index,
        x_features=x_l2,
        num_nodes=data.num_nodes,
        epsilon=epsilon,
        delta=delta,
        clip_norm=gaussian_clip_norm,
        sensitivity_mode=gaussian_sensitivity_mode,
        sensitivity_override=gaussian_sensitivity_override,
    )
    noisy_train_loader, _ = build_node_subgraph_loader(
        nodes=train_nodes_used,
        edge_index=private_edge_index,
        x_features=H_train_noisy,
        labels=data.y,
        query_hops=CONFIG['query_hops'],
        max_graphs=train_nodes_used.numel(),
        batch_size=CONFIG['batch_size'],
        shuffle=True,
        seed=seed + 103,
    )
    noisy_val_loader, _ = build_node_subgraph_loader(
        nodes=val_nodes_used,
        edge_index=val_edge_index,
        x_features=H_val_sgc,
        labels=data.y,
        query_hops=CONFIG['query_hops'],
        max_graphs=val_nodes_used.numel(),
        batch_size=CONFIG['batch_size'],
        shuffle=False,
        seed=seed + 104,
    )
    acc_noisy_sgc_gcn = train_gcn_with_loaders(
        train_loader=noisy_train_loader,
        val_loader=noisy_val_loader,
        num_features=H_train_noisy.size(1),
        num_classes=num_classes,
        epochs=epochs,
        feature_jitter_std=CONFIG['feature_jitter_std'],
        edge_dropout_p=CONFIG['edge_dropout_p'],
    )

    # Baseline B: edge randomized response, same downstream GCN and clean node features.
    rr_edges, p_keep = edge_rr_perturb_edges(
        private_edges=private_edge_index,
        train_nodes_tensor=train_nodes,
        num_nodes=data.num_nodes,
        epsilon=epsilon,
    )
    rr_train_loader, _ = build_node_subgraph_loader(
        nodes=train_nodes_used,
        edge_index=rr_edges,
        x_features=x_l2,
        labels=data.y,
        query_hops=CONFIG['query_hops'],
        max_graphs=train_nodes_used.numel(),
        batch_size=CONFIG['batch_size'],
        shuffle=True,
        seed=seed + 105,
    )
    acc_rr_gcn = train_gcn_with_loaders(
        train_loader=rr_train_loader,
        val_loader=clean_val_loader,
        num_features=x_l2.size(1),
        num_classes=num_classes,
        epochs=epochs,
        feature_jitter_std=CONFIG['feature_jitter_std'],
        edge_dropout_p=CONFIG['edge_dropout_p'],
    )

    return {
        'epsilon_total': float(epsilon),
        'seed': int(seed),
        'baseline_clean_gcn_acc': float(acc_clean_gcn),
        'baseline_noisy_sgc_gcn_acc': float(acc_noisy_sgc_gcn),
        'baseline_rr_gcn_acc': float(acc_rr_gcn),
        'noisy_sgc_sigma': float(noisy_diag['gaussian_sigma']),
        'noisy_sgc_sensitivity': float(noisy_diag['gaussian_sensitivity']),
        'noisy_sgc_snr': float(noisy_diag['snr']),
        'rr_keep_prob': float(p_keep),
        'train_graphs_used': int(train_nodes_used.numel()),
        'val_graphs_used': int(val_nodes_used.numel()),
    }


benchmark_cfg = {
    'eps_list': [1.0, 4.0],
    'seed_list': [int(CONFIG['seed']) + i for i in range(3)],
    'train_cap': min(6000, int(train_nodes.numel())),
    'epochs': min(10, int(CONFIG.get('epochs', 12))),
    'gaussian_clip_norm': 1.0,
    'gaussian_sensitivity_mode': 'moderate',
    'gaussian_sensitivity_override': None,
}

print('Benchmark config:', benchmark_cfg)
benchmark_rows = []
for eps in benchmark_cfg['eps_list']:
    for seed in benchmark_cfg['seed_list']:
        t0 = time.time()
        row = run_fair_edge_dp_baselines_one_seed(
            epsilon=eps,
            delta=1e-5,
            seed=seed,
            train_cap=benchmark_cfg['train_cap'],
            epochs=benchmark_cfg['epochs'],
            gaussian_clip_norm=benchmark_cfg['gaussian_clip_norm'],
            gaussian_sensitivity_mode=benchmark_cfg['gaussian_sensitivity_mode'],
            gaussian_sensitivity_override=benchmark_cfg['gaussian_sensitivity_override'],
        )
        row['elapsed_sec'] = float(time.time() - t0)
        benchmark_rows.append(row)

if pd is not None:
    raw_df = pd.DataFrame(benchmark_rows).sort_values(['epsilon_total', 'seed'])

    baseline_legend_df = pd.DataFrame([
        {
            'metric_key': 'gcn_non_private_clean_upper_bound_(mean/std)',
            'what_is_tested': 'StandardGCN on clean private edges + clean features (non-private utility ceiling).',
        },
        {
            'metric_key': 'gcn_dp_gaussian_feature_(mean/std)',
            'what_is_tested': 'StandardGCN on Gaussian-noised 1-hop features (edge-DP feature mechanism).',
        },
        {
            'metric_key': 'gcn_dp_edge_rr_(mean/std)',
            'what_is_tested': 'StandardGCN on RR-perturbed private edges with clean node features (edge-DP graph mechanism).',
        },
    ])

    agg_df = raw_df.groupby('epsilon_total', as_index=False).agg(
        gcn_non_private_clean_upper_bound_mean=('baseline_clean_gcn_acc', 'mean'),
        gcn_non_private_clean_upper_bound_std=('baseline_clean_gcn_acc', 'std'),
        gcn_dp_gaussian_feature_mean=('baseline_noisy_sgc_gcn_acc', 'mean'),
        gcn_dp_gaussian_feature_std=('baseline_noisy_sgc_gcn_acc', 'std'),
        gcn_dp_edge_rr_mean=('baseline_rr_gcn_acc', 'mean'),
        gcn_dp_edge_rr_std=('baseline_rr_gcn_acc', 'std'),
        noisy_sgc_sigma=('noisy_sgc_sigma', 'mean'),
        noisy_sgc_sensitivity=('noisy_sgc_sensitivity', 'mean'),
        noisy_sgc_snr=('noisy_sgc_snr', 'mean'),
        rr_keep_prob=('rr_keep_prob', 'mean'),
        train_graphs_used=('train_graphs_used', 'mean'),
        val_graphs_used=('val_graphs_used', 'mean'),
        seed_count=('seed', 'nunique'),
        mean_elapsed_sec=('elapsed_sec', 'mean'),
    )

    for col in [
        'gcn_non_private_clean_upper_bound_std',
        'gcn_dp_gaussian_feature_std',
        'gcn_dp_edge_rr_std',
    ]:
        agg_df[col] = agg_df[col].fillna(0.0)

    if 'results' in globals() and isinstance(results, list) and len(results) > 0:
        method_df = (
            pd.DataFrame(results)
            .groupby('epsilon_total', as_index=False)['best_val_acc']
            .mean()
            .rename(columns={'best_val_acc': 'kl_prototype_dp_gcn_mean'})
        )
        compare_df = agg_df.merge(method_df, on='epsilon_total', how='left')
        compare_df['kl_minus_gaussian_feature_dp'] = compare_df['kl_prototype_dp_gcn_mean'] - compare_df['gcn_dp_gaussian_feature_mean']
        compare_df['kl_minus_rr_edge_dp'] = compare_df['kl_prototype_dp_gcn_mean'] - compare_df['gcn_dp_edge_rr_mean']
    else:
        compare_df = agg_df

    print('Benchmark legend (what each GCN metric means):')
    display(baseline_legend_df)
    display(raw_df)
    display(compare_df)

    raw_df.to_csv('edge_dp_fair_benchmark_raw.csv', index=False)
    compare_df.to_csv('edge_dp_fair_benchmark_summary.csv', index=False)
    baseline_legend_df.to_csv('edge_dp_fair_benchmark_legend.csv', index=False)
    print('Saved edge_dp_fair_benchmark_raw.csv')
    print('Saved edge_dp_fair_benchmark_summary.csv')
    print('Saved edge_dp_fair_benchmark_legend.csv')
else:
    print(benchmark_rows)

In [ ]:
# ==========================================
# CELL 7: 4-RUN ACCURACY PLAN (GPU-OPTIMIZED)
# ==========================================

required_vars = [
    'CONFIG', 'data', 'num_classes', 'x_l2',
    'public_nodes', 'train_nodes', 'val_nodes',
    'pub_edge_index', 'private_edge_index', 'val_edge_index',
    'build_class_conditional_public_dictionary', 'synthesize_edge_dp_assignments',
    'build_validation_loader', 'StandardGCN', 'PrototypeAssignmentDataset'
]
missing = [v for v in required_vars if v not in globals()]
if len(missing) > 0:
    raise RuntimeError(f"Missing required variables from earlier cells: {missing}")

if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True
    if hasattr(torch, 'set_float32_matmul_precision'):
        torch.set_float32_matmul_precision('high')


def make_fast_assignment_loader(assignments, public_dict, batch_size=64, shuffle=True):
    ds = PrototypeAssignmentDataset(public_dict, assignments['proto_indices'], assignments['labels'])

    is_cuda = device.type == 'cuda'
    kwargs = {
        'batch_size': batch_size,
        'shuffle': shuffle,
        'num_workers': 2 if is_cuda else 0,
        'pin_memory': is_cuda,
    }
    if kwargs['num_workers'] > 0:
        kwargs['persistent_workers'] = True
        kwargs['prefetch_factor'] = 2

    return DataLoader(ds, **kwargs)


def train_gnn_from_assignments_optimized(
    assignments,
    public_dict,
    val_loader,
    num_features,
    num_classes,
    hidden_channels=32,
    epochs=8,
    batch_size=64,
    lr=3e-3,
    weight_decay=1e-4,
    feature_jitter_std=0.01,
    edge_dropout_p=0.03,
    patience=2,
    min_delta=1e-4,
    grad_clip=1.0,
    compile_model=False,
):
    train_loader = make_fast_assignment_loader(assignments, public_dict, batch_size=batch_size, shuffle=True)

    model = StandardGCN(
        hidden_channels=hidden_channels,
        num_features=num_features,
        num_classes=num_classes,
    ).to(device)

    if compile_model and device.type == 'cuda' and hasattr(torch, 'compile'):
        try:
            model = torch.compile(model)
        except Exception:
            pass

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(epochs, 1))
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

    use_cuda_amp = device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_cuda_amp)

    best_val = 0.0
    best_state = None
    stale_epochs = 0

    for epoch in range(1, epochs + 1):
        model.train()
        for batch in train_loader:
            batch = batch.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            x_aug = batch.x + torch.randn_like(batch.x) * feature_jitter_std
            edge_index_aug, _ = dropout_edge(batch.edge_index, p=edge_dropout_p)
            if edge_index_aug.numel() == 0:
                edge_index_aug = batch.edge_index

            if use_cuda_amp:
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    out = model(x_aug, edge_index_aug, batch.batch)
                    loss = criterion(out, batch.y)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                out = model(x_aug, edge_index_aug, batch.batch)
                loss = criterion(out, batch.y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()

        scheduler.step()

        model.eval()
        val_correct = 0
        with torch.no_grad():
            for vb in val_loader:
                vb = vb.to(device, non_blocking=True)
                pred = model(vb.x, vb.edge_index, vb.batch).argmax(dim=1)
                val_correct += int((pred == vb.y).sum())

        val_acc = val_correct / max(len(val_loader.dataset), 1)
        improved = val_acc > (best_val + min_delta)
        if improved:
            best_val = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale_epochs = 0
        else:
            stale_epochs += 1

        if epoch == 1 or epoch % 2 == 0:
            print(f"  Epoch {epoch:02d} | val_acc={val_acc:.4f} | best={best_val:.4f} | stale={stale_epochs}")

        if stale_epochs >= patience:
            print(f"  Early stop at epoch {epoch} (patience={patience}).")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return float(best_val)


def run_accuracy_plan_variant(run_name, cfg_overrides):
    cfg = dict(CONFIG)
    cfg.update(cfg_overrides)

    set_seed(int(cfg['seed']))

    t0 = time.time()

    # Rebuild dictionary for each run so mechanism-level changes are reflected.
    public_dict_local, public_proto_feats_local, class_to_proto_local = build_class_conditional_public_dictionary(
        data_obj=data,
        x_features=x_l2,
        labels_tensor=data.y,
        public_nodes_tensor=public_nodes,
        pub_edges=pub_edge_index,
        num_classes=num_classes,
        dict_per_class=int(cfg['dict_per_class']),
        num_hops=int(cfg['walk_hops']),
        query_mode=cfg['query_mode'],
        min_class_fraction=float(cfg['min_class_fraction']),
        max_proto_nodes=int(cfg['max_proto_nodes']),
    )

    val_loader_local, _ = build_validation_loader(
        x_l2=x_l2,
        labels=data.y,
        val_nodes=val_nodes,
        val_edge_index=val_edge_index,
        query_hops=int(cfg['query_hops']),
        max_val_graphs=int(cfg['max_val_nodes']),
        seed=int(cfg['seed']) + 33,
    )

    utility_sensitivity = 2.0 if cfg['query_mode'] == 'two_hop_concat' else 1.0

    assignments, diag = synthesize_edge_dp_assignments(
        private_edges=private_edge_index,
        x_features=x_l2,
        labels=data.y,
        private_nodes_tensor=train_nodes,
        dict_features=public_proto_feats_local,
        class_to_proto_indices=class_to_proto_local,
        epsilon_total=float(cfg['target_epsilon']),
        utility_sensitivity=utility_sensitivity,
        query_mode=cfg['query_mode'],
        label_conditioning=bool(cfg['label_conditioning']),
    )

    best_val = train_gnn_from_assignments_optimized(
        assignments=assignments,
        public_dict=public_dict_local,
        val_loader=val_loader_local,
        num_features=data.x.size(1),
        num_classes=num_classes,
        hidden_channels=int(cfg['hidden_channels']),
        epochs=int(cfg['epochs']),
        batch_size=int(cfg['batch_size']),
        lr=float(cfg['lr']),
        weight_decay=float(cfg['weight_decay']),
        feature_jitter_std=float(cfg['feature_jitter_std']),
        edge_dropout_p=float(cfg['edge_dropout_p']),
        patience=int(cfg['patience']),
        min_delta=float(cfg['min_delta']),
        grad_clip=float(cfg['grad_clip']),
        compile_model=bool(cfg['compile_model']),
    )

    elapsed = time.time() - t0

    if device.type == 'cuda':
        torch.cuda.empty_cache()

    return {
        'run_name': run_name,
        'target_epsilon': float(cfg['target_epsilon']),
        'best_val_acc': float(best_val),
        'hidden_channels': int(cfg['hidden_channels']),
        'lr': float(cfg['lr']),
        'feature_jitter_std': float(cfg['feature_jitter_std']),
        'edge_dropout_p': float(cfg['edge_dropout_p']),
        'query_mode': str(cfg['query_mode']),
        'dict_per_class': int(cfg['dict_per_class']),
        'walk_hops': int(cfg['walk_hops']),
        'mean_entropy': float(diag['mean_entropy']),
        'mean_top_probability': float(diag['mean_top_probability']),
        'unique_selected_ratio': float(diag['unique_selected_ratio']),
        'dict_size': int(diag['dict_size']),
        'elapsed_sec': float(elapsed),
    }


base_opt = {
    'target_epsilon': 4.0,
    'lr': 3e-3,
    'weight_decay': 1e-4,
    'feature_jitter_std': 0.01,
    'edge_dropout_p': 0.03,
    'patience': 10,
    'min_delta': 1e-4,
    'grad_clip': 1.0,
    'compile_model': False,
    'hidden_channels': 32,
    'query_mode': 'one_hop',
    'dict_per_class': 48,
    'walk_hops': 1,
}

PLAN_RUNS = [
    ('run1_tuned_training', dict(base_opt)),
    ('run2_tuned_plus_hidden64', {**base_opt, 'hidden_channels': 64}),
    ('run3_plus_two_hop_query', {**base_opt, 'hidden_channels': 64, 'query_mode': 'two_hop_concat'}),
    ('run4_plus_bigger_dictionary', {**base_opt, 'hidden_channels': 64, 'query_mode': 'two_hop_concat', 'dict_per_class': 64, 'walk_hops': 2}),
]

print('Executing 4-run accuracy plan with config deltas:')
for name, cfg_delta in PLAN_RUNS:
    print(f"  - {name}: eps={cfg_delta['target_epsilon']} | hidden={cfg_delta['hidden_channels']} | mode={cfg_delta['query_mode']} | dict/class={cfg_delta['dict_per_class']} | walk_hops={cfg_delta['walk_hops']}")

plan_rows = []
for run_name, cfg_delta in PLAN_RUNS:
    print('\n' + '=' * 72)
    print(f'Running {run_name}')
    row = run_accuracy_plan_variant(run_name, cfg_delta)
    plan_rows.append(row)
    print(f"{run_name} -> best_val_acc={row['best_val_acc']:.4f} | entropy={row['mean_entropy']:.4f} | top_p={row['mean_top_probability']:.4f} | elapsed={row['elapsed_sec']:.1f}s")

if pd is not None:
    plan_df = pd.DataFrame(plan_rows).sort_values('best_val_acc', ascending=False)
    display(plan_df)
    plan_df.to_csv('ogbn_arxiv_4run_accuracy_plan_gpu.csv', index=False)
    print('Saved ogbn_arxiv_4run_accuracy_plan_gpu.csv')
else:
    print(plan_rows)

In [6]:
def train_gnn_from_assignments_optimized(
    assignments,
    public_dict,
    val_loader,
    num_features,
    num_classes,
    hidden_channels=32,
    epochs=8,
    batch_size=64,
    lr=0.01,
    weight_decay=1e-4,
    feature_jitter_std=0.01,
    edge_dropout_p=0.03,
    patience=2,
    min_delta=1e-4,
    grad_clip=1.0,
    compile_model=False,
):
    train_loader = make_fast_assignment_loader(assignments, public_dict, batch_size=batch_size, shuffle=True)

    model = StandardGCN(
        hidden_channels=hidden_channels,
        num_features=num_features,
        num_classes=num_classes,
    ).to(device)

    # Swapped for SGD with Nesterov momentum
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=lr,
        momentum=0.9,
        nesterov=True,
        weight_decay=weight_decay
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(epochs, 1))
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

    use_cuda_amp = device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_cuda_amp)

    best_val = 0.0
    best_state = None
    stale_epochs = 0

    for epoch in range(1, epochs + 1):
        model.train()
        for batch in train_loader:
            batch = batch.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            x_aug = batch.x + torch.randn_like(batch.x) * feature_jitter_std
            edge_index_aug, _ = dropout_edge(batch.edge_index, p=edge_dropout_p)
            if edge_index_aug.numel() == 0:
                edge_index_aug = batch.edge_index

            if use_cuda_amp:
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    out = model(x_aug, edge_index_aug, batch.batch)
                    loss = criterion(out, batch.y)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                out = model(x_aug, edge_index_aug, batch.batch)
                loss = criterion(out, batch.y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()

        scheduler.step()

        model.eval()
        val_correct = 0
        with torch.no_grad():
            for vb in val_loader:
                vb = vb.to(device, non_blocking=True)
                pred = model(vb.x, vb.edge_index, vb.batch).argmax(dim=1)
                val_correct += int((pred == vb.y).sum())

        val_acc = val_correct / max(len(val_loader.dataset), 1)
        improved = val_acc > (best_val + min_delta)
        if improved:
            best_val = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale_epochs = 0
        else:
            stale_epochs += 1

        if epoch == 1 or epoch % 10 == 0:
            print(f"  Epoch {epoch:02d} | val_acc={val_acc:.4f} | best={best_val:.4f} | stale={stale_epochs}")

        if stale_epochs >= patience:
            print(f"  Early stop at epoch {epoch}.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return float(best_val)


In [ ]:
```python
def train_gnn_from_assignments_optimized(
    assignments,
    public_dict,
    val_loader,
    num_features,
    num_classes,
    hidden_channels=32,
    epochs=8,
    batch_size=64,
    lr=0.01,  # Standard starting LR for SGD
    weight_decay=1e-4,
    feature_jitter_std=0.01,
    edge_dropout_p=0.03,
    patience=2,
    min_delta=1e-4,
    grad_clip=1.0,
    compile_model=False,
):
    train_loader = make_fast_assignment_loader(assignments, public_dict, batch_size=batch_size, shuffle=True)

    model = StandardGCN(
        hidden_channels=hidden_channels,
        num_features=num_features,
        num_classes=num_classes,
    ).to(device)

    if compile_model and device.type == 'cuda' and hasattr(torch, 'compile'):
        try:
            model = torch.compile(model)
        except Exception:
            pass

    # Swapped for SGD with Nesterov momentum
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=lr,
        momentum=0.9,
        nesterov=True,
        weight_decay=weight_decay
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(epochs, 1))
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

    use_cuda_amp = device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_cuda_amp)

    best_val = 0.0
    best_state = None
    stale_epochs = 0

    for epoch in range(1, epochs + 1):
        model.train()
        for batch in train_loader:
            batch = batch.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            x_aug = batch.x + torch.randn_like(batch.x) * feature_jitter_std
            edge_index_aug, _ = dropout_edge(batch.edge_index, p=edge_dropout_p)
            if edge_index_aug.numel() == 0:
                edge_index_aug = batch.edge_index

            if use_cuda_amp:
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    out = model(x_aug, edge_index_aug, batch.batch)
                    loss = criterion(out, batch.y)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                out = model(x_aug, edge_index_aug, batch.batch)
                loss = criterion(out, batch.y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()

        scheduler.step()

        model.eval()
        val_correct = 0
        with torch.no_grad():
            for vb in val_loader:
                vb = vb.to(device, non_blocking=True)
                pred = model(vb.x, vb.edge_index, vb.batch).argmax(dim=1)
                val_correct += int((pred == vb.y).sum())

        val_acc = val_correct / max(len(val_loader.dataset), 1)
        improved = val_acc > (best_val + min_delta)
        if improved:
            best_val = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale_epochs = 0
        else:
            stale_epochs += 1

        if epoch == 1 or epoch % 2 == 0:
            print(f"  Epoch {epoch:02d} | val_acc={val_acc:.4f} | best={best_val:.4f} | stale={stale_epochs}")

        if stale_epochs >= patience:
            print(f"  Early stop at epoch {epoch} (patience={patience}).")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return float(best_val)
```

In [ ]:
# ==========================================
# CELL 8: PAPER-DEFAULT PROTOCOL + FIRST-5 LAUNCHER
# ==========================================

import json
from pathlib import Path
from datetime import datetime


PROTOCOL_REL_PATH = Path('configs/paper_protocol_v1.json')
PROJECT_ROOT_CANDIDATES = [
    Path.cwd(),
    Path.cwd() / 'final_project' / 'Clinical-DP',
    Path.cwd().parent / 'final_project' / 'Clinical-DP',
]
PROTOCOL_CANDIDATES = []
for root in PROJECT_ROOT_CANDIDATES:
    PROTOCOL_CANDIDATES.append(root / PROTOCOL_REL_PATH)
PROTOCOL_CANDIDATES.extend([
    Path('configs/paper_protocol_v1.json'),
    Path('final_project/Clinical-DP/configs/paper_protocol_v1.json'),
])

PROTOCOL_FILE = next((p for p in PROTOCOL_CANDIDATES if p.exists()), None)
if PROTOCOL_FILE is None:
    DEFAULT_PROTOCOL = {
        'protocol_name': 'paper_default_v1',
        'dataset': {
            'name': 'ogbn-arxiv',
            'split_mode': 'official_ogb',
            'public_private_partition_rule': 'public_from_train_stratified',
            'public_frac': 0.02,
        },
        'caps': {
            'max_private_nodes': 6000,
            'max_val_nodes': 3000,
            'max_test_nodes': 3000,
        },
        'mechanism': {
            'query_mode': 'one_hop',
            'query_normalization': 'none',
            'query_clip_norm': 1.0,
            'walk_hops': 1,
            'query_hops': 1,
            'dict_per_class': 32,
            'max_proto_nodes': 64,
            'min_class_fraction': 1.0,
            'label_conditioning': True,
        },
        'downstream_budget': {
            'hidden_channels': 32,
            'epochs': 10,
            'batch_size': 64,
            'lr': 0.01,
            'feature_jitter_std': 0.03,
            'edge_dropout_p': 0.10,
        },
        'tracks': {
            'defensible': {
                'privacy_claim': 'defensible',
                'query_mode': 'one_hop',
                'query_normalization': 'none',
            },
            'exploratory': {
                'privacy_claim': 'exploratory',
                'query_mode': 'two_hop_concat',
                'query_normalization': 'l2_row',
            },
        },
        'baseline_labels': {
            'baseline_clean_upper_bound': 'non_private_upper_bound',
            'baseline_gaussian_practical': 'practical_not_theorem_tight',
            'baseline_rr_practical': 'practical_not_theorem_tight',
        },
        'default_launch': {
            'seeds': [42, 43, 44],
            'epsilons': [0.5, 1.0, 4.0],
        },
    }
    notebook_root = next((r for r in PROJECT_ROOT_CANDIDATES if (r / 'KL_perturb2.ipynb').exists()), Path.cwd())
    PROTOCOL_FILE = notebook_root / PROTOCOL_REL_PATH
    PROTOCOL_FILE.parent.mkdir(parents=True, exist_ok=True)
    with PROTOCOL_FILE.open('w') as f:
        json.dump(DEFAULT_PROTOCOL, f, indent=2)
    print(f'[info] Protocol file was missing; created default at {PROTOCOL_FILE.resolve()}')

with PROTOCOL_FILE.open('r') as f:
    CANONICAL_PROTOCOL = json.load(f)


def _deep_update(base_dict, update_dict):
    for k, v in update_dict.items():
        if isinstance(v, dict) and isinstance(base_dict.get(k), dict):
            _deep_update(base_dict[k], v)
        else:
            base_dict[k] = v
    return base_dict


def build_protocol(track='defensible', overrides=None):
    protocol = json.loads(json.dumps(CANONICAL_PROTOCOL))

    if track not in protocol['tracks']:
        raise ValueError(f'Unknown track: {track}')

    track_cfg = protocol['tracks'][track]
    protocol['privacy_track'] = str(track)
    protocol['privacy_claim'] = str(track_cfg['privacy_claim'])
    protocol['mechanism']['query_mode'] = str(track_cfg['query_mode'])
    protocol['mechanism']['query_normalization'] = str(track_cfg['query_normalization'])

    if overrides is not None:
        _deep_update(protocol, overrides)

    return protocol


def stratified_public_private_from_nodes(nodes, labels, public_frac=0.02, seed=0):
    g = torch.Generator().manual_seed(seed)

    labels_nodes = labels[nodes]
    classes = torch.unique(labels_nodes)

    public_parts = []
    private_parts = []
    for c in classes.tolist():
        c_nodes = nodes[labels_nodes == c]
        if c_nodes.numel() == 0:
            continue

        perm = c_nodes[torch.randperm(c_nodes.numel(), generator=g)]
        n_pub = max(1, int(float(public_frac) * c_nodes.numel()))
        n_pub = min(n_pub, c_nodes.numel())

        public_parts.append(perm[:n_pub])
        private_parts.append(perm[n_pub:])

    public_nodes = torch.cat(public_parts) if len(public_parts) > 0 else torch.tensor([], dtype=torch.long)
    private_nodes = torch.cat(private_parts) if len(private_parts) > 0 else torch.tensor([], dtype=torch.long)

    if public_nodes.numel() > 0:
        public_nodes = public_nodes[torch.randperm(public_nodes.numel(), generator=g)]
    if private_nodes.numel() > 0:
        private_nodes = private_nodes[torch.randperm(private_nodes.numel(), generator=g)]

    return public_nodes, private_nodes


def resolve_split_bundle(dataset_obj, data_obj, protocol, seed):
    split_mode = protocol['dataset']['split_mode']
    public_frac = float(protocol['dataset']['public_frac'])

    if split_mode == 'official_ogb':
        if not hasattr(dataset_obj, 'get_idx_split'):
            raise RuntimeError('official_ogb split requested but dataset has no get_idx_split().')

        split_idx = dataset_obj.get_idx_split()
        train_pool = split_idx['train'].view(-1).long()
        val_nodes = split_idx['valid'].view(-1).long()
        test_nodes = split_idx['test'].view(-1).long()

        public_nodes, private_nodes = stratified_public_private_from_nodes(
            train_pool,
            data_obj.y,
            public_frac=public_frac,
            seed=seed + 11,
        )
    elif split_mode == 'custom_stratified':
        public_nodes, private_nodes, val_nodes = stratified_split_indices(
            data_obj.y,
            public_frac=public_frac,
            val_frac=float(CONFIG.get('val_frac', 0.02)),
            seed=seed + 11,
        )
        test_nodes = val_nodes.clone()
    else:
        raise ValueError(f'Unsupported split_mode: {split_mode}')

    caps = protocol['caps']
    private_nodes = cap_indices(private_nodes, int(caps['max_private_nodes']), seed=seed + 12)
    val_nodes = cap_indices(val_nodes, int(caps['max_val_nodes']), seed=seed + 13)
    test_nodes = cap_indices(test_nodes, int(caps['max_test_nodes']), seed=seed + 14)

    pub_edge_index, private_edge_index, val_edge_index = split_edges_by_nodes(
        data_obj.edge_index,
        public_nodes,
        private_nodes,
        val_nodes,
        data_obj.num_nodes,
    )

    return {
        'split_mode': split_mode,
        'public_nodes': public_nodes,
        'private_nodes': private_nodes,
        'val_nodes': val_nodes,
        'test_nodes': test_nodes,
        'pub_edge_index': pub_edge_index,
        'private_edge_index': private_edge_index,
        'val_edge_index': val_edge_index,
    }


def _to_serializable(obj):
    if isinstance(obj, torch.Tensor):
        if obj.ndim == 0:
            return obj.item()
        return obj.detach().cpu().tolist()

    if isinstance(obj, dict):
        return {str(k): _to_serializable(v) for k, v in obj.items()}

    if isinstance(obj, (list, tuple)):
        return [_to_serializable(x) for x in obj]

    if hasattr(obj, 'item') and callable(getattr(obj, 'item')):
        try:
            return obj.item()
        except Exception:
            return str(obj)

    return obj


def get_git_commit_hash(repo_path='.'):
    try:
        out = subprocess.check_output(
            ['git', 'rev-parse', 'HEAD'],
            cwd=repo_path,
            stderr=subprocess.DEVNULL,
        )
        return out.decode().strip()
    except Exception:
        return 'unknown'


def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w') as f:
        json.dump(_to_serializable(payload), f, indent=2)


def run_prototype_one_seed_epsilon(data_obj, protocol, split_bundle, epsilon, seed, artifact_root):
    mech = protocol['mechanism']
    down = protocol['downstream_budget']

    set_seed(seed)
    t0 = time.time()

    public_dict, public_proto_feats, class_to_proto_indices, dict_diag = build_class_conditional_public_dictionary(
        data_obj=data_obj,
        x_features=data_obj.x,
        labels_tensor=data_obj.y,
        public_nodes_tensor=split_bundle['public_nodes'],
        pub_edges=split_bundle['pub_edge_index'],
        num_classes=num_classes,
        dict_per_class=int(mech['dict_per_class']),
        num_hops=int(mech['walk_hops']),
        query_mode=mech['query_mode'],
        min_class_fraction=float(mech['min_class_fraction']),
        max_proto_nodes=int(mech['max_proto_nodes']),
        return_diagnostics=True,
        rng_seed=seed + 21,
    )

    dict_fit_diag = analyze_dictionary_query_fit(
        private_edges=split_bundle['private_edge_index'],
        x_features=data_obj.x,
        labels=data_obj.y,
        private_nodes_tensor=split_bundle['private_nodes'],
        dict_features=public_proto_feats,
        class_to_proto_indices=class_to_proto_indices,
        query_mode=mech['query_mode'],
        query_normalization=mech['query_normalization'],
        query_clip_norm=float(mech['query_clip_norm']),
        max_nodes_per_class=2000,
    )

    val_loader, val_used = build_validation_loader(
        x_l2=data_obj.x,
        labels=data_obj.y,
        val_nodes=split_bundle['val_nodes'],
        val_edge_index=split_bundle['val_edge_index'],
        query_hops=int(mech['query_hops']),
        max_val_graphs=int(protocol['caps']['max_val_nodes']),
        seed=seed + 22,
    )

    utility_sensitivity = 2.0 if mech['query_mode'] == 'two_hop_concat' else 1.0
    assignments, assign_diag = synthesize_edge_dp_assignments(
        private_edges=split_bundle['private_edge_index'],
        x_features=data_obj.x,
        labels=data_obj.y,
        private_nodes_tensor=split_bundle['private_nodes'],
        dict_features=public_proto_feats,
        class_to_proto_indices=class_to_proto_indices,
        epsilon_total=float(epsilon),
        utility_sensitivity=utility_sensitivity,
        query_mode=mech['query_mode'],
        label_conditioning=bool(mech['label_conditioning']),
        query_normalization=mech['query_normalization'],
        query_clip_norm=float(mech['query_clip_norm']),
        return_full_diagnostics=True,
    )

    best_val_acc = train_gnn_from_assignments(
        assignments=assignments,
        public_dict=public_dict,
        val_loader=val_loader,
        num_features=data_obj.x.size(1),
        num_classes=num_classes,
        epochs=int(down['epochs']),
        batch_size=int(down['batch_size']),
        lr=float(down['lr']),
        feature_jitter_std=float(down['feature_jitter_std']),
        edge_dropout_p=float(down['edge_dropout_p']),
    )

    elapsed = float(time.time() - t0)

    run_row = {
        'method': 'kl_prototype_dp',
        'privacy_track': str(protocol['privacy_track']),
        'privacy_claim': str(protocol['privacy_claim']),
        'split_mode': str(split_bundle['split_mode']),
        'epsilon_total': float(epsilon),
        'seed': int(seed),
        'best_val_acc': float(best_val_acc),
        'dict_size': int(assign_diag['dict_size']),
        'mean_entropy': float(assign_diag['mean_entropy']),
        'mean_top_probability': float(assign_diag['mean_top_probability']),
        'mean_nearest_distance': float(assign_diag['mean_nearest_distance']),
        'mean_sampled_distance': float(assign_diag['mean_sampled_distance']),
        'unique_selected_ratio': float(assign_diag['unique_selected_ratio']),
        'unused_prototype_fraction': float(assign_diag['unused_prototype_fraction']),
        'class_coverage_selected_prototypes': float(assign_diag['class_coverage_selected_prototypes']),
        'train_graphs_used': int(split_bundle['private_nodes'].numel()),
        'val_graphs_used': int(val_used.numel()),
        'elapsed_sec': elapsed,
    }

    run_stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    run_dir = Path(artifact_root) / f"prototype_eps{epsilon}_seed{seed}_{run_stamp}"
    run_dir.mkdir(parents=True, exist_ok=True)

    run_config = {
        'protocol': protocol,
        'seed': int(seed),
        'epsilon_total': float(epsilon),
        'git_commit_hash': get_git_commit_hash('.'),
        'timestamp': run_stamp,
    }

    write_json(run_dir / 'config.json', run_config)
    write_json(run_dir / 'metrics.json', run_row)
    write_json(run_dir / 'assignment_diagnostics.json', assign_diag)
    write_json(run_dir / 'dictionary_diagnostics.json', dict_diag)
    write_json(run_dir / 'dictionary_query_fit.json', dict_fit_diag)

    return run_row


def aggregate_mean_std(df, metric_col, group_col='epsilon_total'):
    out = df.groupby(group_col, as_index=False).agg(
        mean=(metric_col, 'mean'),
        std=(metric_col, 'std'),
        seed_count=('seed', 'nunique'),
    )
    out['std'] = out['std'].fillna(0.0)
    return out


def run_prototype_grid(dataset_obj, data_obj, protocol, epsilons, seeds, artifact_root):
    rows = []

    for seed in seeds:
        split_bundle = resolve_split_bundle(dataset_obj, data_obj, protocol, seed)
        for eps in epsilons:
            print('\n' + '=' * 72)
            print(
                f"Prototype run | track={protocol['privacy_track']} | split={split_bundle['split_mode']} | seed={seed} | eps={eps}"
            )
            row = run_prototype_one_seed_epsilon(
                data_obj=data_obj,
                protocol=protocol,
                split_bundle=split_bundle,
                epsilon=float(eps),
                seed=int(seed),
                artifact_root=artifact_root,
            )
            rows.append(row)
            print(
                f"val_acc={row['best_val_acc']:.4f} | entropy={row['mean_entropy']:.4f} | top_p={row['mean_top_probability']:.4f} | elapsed={row['elapsed_sec']:.1f}s"
            )

    if pd is None:
        return rows, None

    raw_df = pd.DataFrame(rows).sort_values(['epsilon_total', 'seed'])
    summary_df = raw_df.groupby('epsilon_total', as_index=False).agg(
        mean_acc=('best_val_acc', 'mean'),
        std_acc=('best_val_acc', 'std'),
        mean_entropy=('mean_entropy', 'mean'),
        mean_top_probability=('mean_top_probability', 'mean'),
        mean_unique_selected_ratio=('unique_selected_ratio', 'mean'),
        mean_elapsed_sec=('elapsed_sec', 'mean'),
        train_graphs_used=('train_graphs_used', 'mean'),
        val_graphs_used=('val_graphs_used', 'mean'),
        privacy_track=('privacy_track', 'first'),
        split_mode=('split_mode', 'first'),
        seed_count=('seed', 'nunique'),
    )
    summary_df['std_acc'] = summary_df['std_acc'].fillna(0.0)

    out_dir = Path(artifact_root)
    out_dir.mkdir(parents=True, exist_ok=True)
    raw_df.to_csv(out_dir / 'prototype_raw.csv', index=False)
    summary_df.to_csv(out_dir / 'prototype_summary.csv', index=False)

    return raw_df, summary_df


def run_matched_baseline_grid(protocol, epsilons, seeds, artifact_root):
    if 'run_fair_edge_dp_baselines_one_seed' not in globals():
        raise RuntimeError('Baseline runner not found. Execute Cell 6 first to define it.')

    rows = []
    train_cap = int(protocol['caps']['max_private_nodes'])
    epochs = int(protocol['downstream_budget']['epochs'])
    batch_size = int(protocol['downstream_budget']['batch_size'])

    old_batch_size = CONFIG.get('batch_size', batch_size)
    CONFIG['batch_size'] = batch_size
    try:
        for eps in epsilons:
            for seed in seeds:
                print('\n' + '-' * 72)
                print(f"Baseline run | seed={seed} | eps={eps} | train_cap={train_cap} | epochs={epochs}")
                t0 = time.time()
                r = run_fair_edge_dp_baselines_one_seed(
                    epsilon=float(eps),
                    delta=1e-5,
                    seed=int(seed),
                    train_cap=train_cap,
                    epochs=epochs,
                )
                r['elapsed_sec'] = float(time.time() - t0)
                r['privacy_track'] = str(protocol['privacy_track'])
                r['privacy_claim'] = str(protocol['privacy_claim'])
                r['split_mode'] = str(protocol['dataset']['split_mode'])
                rows.append(r)
    finally:
        CONFIG['batch_size'] = old_batch_size

    if pd is None:
        return rows, None, None

    raw_df = pd.DataFrame(rows).sort_values(['epsilon_total', 'seed'])

    long_rows = []
    for _, rr in raw_df.iterrows():
        long_rows.append({
            'epsilon_total': rr['epsilon_total'],
            'seed': rr['seed'],
            'method': 'baseline_clean_upper_bound',
            'acc': rr['baseline_clean_gcn_acc'],
            'train_graphs_used': rr['train_graphs_used'],
            'val_graphs_used': rr['val_graphs_used'],
            'elapsed_sec': rr['elapsed_sec'],
            'privacy_track': rr['privacy_track'],
            'split_mode': rr['split_mode'],
        })
        long_rows.append({
            'epsilon_total': rr['epsilon_total'],
            'seed': rr['seed'],
            'method': 'baseline_gaussian_practical',
            'acc': rr['baseline_noisy_sgc_gcn_acc'],
            'train_graphs_used': rr['train_graphs_used'],
            'val_graphs_used': rr['val_graphs_used'],
            'elapsed_sec': rr['elapsed_sec'],
            'privacy_track': rr['privacy_track'],
            'split_mode': rr['split_mode'],
        })
        long_rows.append({
            'epsilon_total': rr['epsilon_total'],
            'seed': rr['seed'],
            'method': 'baseline_rr_practical',
            'acc': rr['baseline_rr_gcn_acc'],
            'train_graphs_used': rr['train_graphs_used'],
            'val_graphs_used': rr['val_graphs_used'],
            'elapsed_sec': rr['elapsed_sec'],
            'privacy_track': rr['privacy_track'],
            'split_mode': rr['split_mode'],
        })

    baseline_long_df = pd.DataFrame(long_rows)
    baseline_summary_df = baseline_long_df.groupby(['epsilon_total', 'method'], as_index=False).agg(
        mean_acc=('acc', 'mean'),
        std_acc=('acc', 'std'),
        train_graphs_used=('train_graphs_used', 'mean'),
        val_graphs_used=('val_graphs_used', 'mean'),
        mean_elapsed_sec=('elapsed_sec', 'mean'),
        privacy_track=('privacy_track', 'first'),
        split_mode=('split_mode', 'first'),
        seed_count=('seed', 'nunique'),
    )
    baseline_summary_df['std_acc'] = baseline_summary_df['std_acc'].fillna(0.0)

    out_dir = Path(artifact_root)
    out_dir.mkdir(parents=True, exist_ok=True)
    raw_df.to_csv(out_dir / 'baseline_raw.csv', index=False)
    baseline_long_df.to_csv(out_dir / 'baseline_long.csv', index=False)
    baseline_summary_df.to_csv(out_dir / 'baseline_summary.csv', index=False)

    return raw_df, baseline_long_df, baseline_summary_df


def build_first_five_specs():
    seeds = [42, 43, 44]
    eps = [0.5, 1.0, 4.0]

    specs = [
        {
            'name': 'exp1_one_hop_no_norm_matched6k',
            'track': 'defensible',
            'protocol_overrides': {
                'dataset': {'split_mode': 'custom_stratified'},
                'caps': {'max_private_nodes': 6000, 'max_val_nodes': 3000, 'max_test_nodes': 3000},
                'mechanism': {'query_mode': 'one_hop', 'query_normalization': 'none'},
            },
            'epsilons': eps,
            'seeds': seeds,
        },
        {
            'name': 'exp2_one_hop_l2_row_norm_matched6k',
            'track': 'defensible',
            'protocol_overrides': {
                'dataset': {'split_mode': 'custom_stratified'},
                'caps': {'max_private_nodes': 6000, 'max_val_nodes': 3000, 'max_test_nodes': 3000},
                'mechanism': {'query_mode': 'one_hop', 'query_normalization': 'l2_row'},
            },
            'epsilons': eps,
            'seeds': seeds,
        },
        {
            'name': 'exp3_one_hop_dict_size_sweep_matched6k',
            'track': 'defensible',
            'sweep_key': 'mechanism.dict_per_class',
            'sweep_values': [8, 16, 32, 48, 64, 96],
            'protocol_overrides': {
                'dataset': {'split_mode': 'custom_stratified'},
                'caps': {'max_private_nodes': 6000, 'max_val_nodes': 3000, 'max_test_nodes': 3000},
                'mechanism': {'query_mode': 'one_hop', 'query_normalization': 'none'},
            },
            'epsilons': [1.0, 4.0],
            'seeds': seeds,
        },
        {
            'name': 'exp4_one_hop_public_frac_sweep_matched6k',
            'track': 'defensible',
            'sweep_key': 'dataset.public_frac',
            'sweep_values': [0.005, 0.01, 0.02, 0.05],
            'protocol_overrides': {
                'dataset': {'split_mode': 'custom_stratified'},
                'caps': {'max_private_nodes': 6000, 'max_val_nodes': 3000, 'max_test_nodes': 3000},
                'mechanism': {'query_mode': 'one_hop', 'query_normalization': 'none'},
            },
            'epsilons': [1.0, 4.0],
            'seeds': seeds,
        },
        {
            'name': 'exp5_official_ogb_pilot_best_one_hop',
            'track': 'defensible',
            'protocol_overrides': {
                'dataset': {'split_mode': 'official_ogb'},
                'caps': {'max_private_nodes': 6000, 'max_val_nodes': 3000, 'max_test_nodes': 3000},
                'mechanism': {'query_mode': 'one_hop', 'query_normalization': 'none'},
            },
            'epsilons': [1.0, 4.0],
            'seeds': seeds,
        },
    ]

    return specs


def _set_nested_value(root, dotted_key, value):
    keys = dotted_key.split('.')
    cur = root
    for k in keys[:-1]:
        cur = cur[k]
    cur[keys[-1]] = value


def launch_first_five_experiments(dataset_obj, data_obj, execute=False, artifact_root='artifacts/paper_default_v1'):
    specs = build_first_five_specs()

    if not execute:
        plan_rows = []
        for s in specs:
            if 'sweep_key' in s:
                for sv in s['sweep_values']:
                    plan_rows.append({
                        'experiment': s['name'],
                        'track': s['track'],
                        'sweep_key': s['sweep_key'],
                        'sweep_value': sv,
                        'split_mode': s['protocol_overrides']['dataset']['split_mode'],
                        'epsilons': s['epsilons'],
                        'seeds': s['seeds'],
                    })
            else:
                plan_rows.append({
                    'experiment': s['name'],
                    'track': s['track'],
                    'sweep_key': None,
                    'sweep_value': None,
                    'split_mode': s['protocol_overrides']['dataset']['split_mode'],
                    'epsilons': s['epsilons'],
                    'seeds': s['seeds'],
                })

        if pd is not None:
            return pd.DataFrame(plan_rows)
        return plan_rows

    all_raw = []
    all_summary = []

    for spec in specs:
        protocol = build_protocol(track=spec['track'], overrides=spec['protocol_overrides'])

        if 'sweep_key' in spec:
            for sweep_value in spec['sweep_values']:
                p_run = json.loads(json.dumps(protocol))
                _set_nested_value(p_run, spec['sweep_key'], sweep_value)

                run_root = Path(artifact_root) / spec['name'] / f"{spec['sweep_key'].replace('.', '_')}={sweep_value}"
                raw_df, summary_df = run_prototype_grid(
                    dataset_obj=dataset_obj,
                    data_obj=data_obj,
                    protocol=p_run,
                    epsilons=spec['epsilons'],
                    seeds=spec['seeds'],
                    artifact_root=run_root,
                )

                if pd is not None and raw_df is not None:
                    raw_df['experiment'] = spec['name']
                    raw_df['sweep_key'] = spec['sweep_key']
                    raw_df['sweep_value'] = sweep_value
                    all_raw.append(raw_df)

                if pd is not None and summary_df is not None:
                    summary_df['experiment'] = spec['name']
                    summary_df['sweep_key'] = spec['sweep_key']
                    summary_df['sweep_value'] = sweep_value
                    all_summary.append(summary_df)
        else:
            run_root = Path(artifact_root) / spec['name']
            raw_df, summary_df = run_prototype_grid(
                dataset_obj=dataset_obj,
                data_obj=data_obj,
                protocol=protocol,
                epsilons=spec['epsilons'],
                seeds=spec['seeds'],
                artifact_root=run_root,
            )

            if pd is not None and raw_df is not None:
                raw_df['experiment'] = spec['name']
                raw_df['sweep_key'] = None
                raw_df['sweep_value'] = None
                all_raw.append(raw_df)

            if pd is not None and summary_df is not None:
                summary_df['experiment'] = spec['name']
                summary_df['sweep_key'] = None
                summary_df['sweep_value'] = None
                all_summary.append(summary_df)

    if pd is None:
        return None, None

    master_raw = pd.concat(all_raw, ignore_index=True) if len(all_raw) > 0 else pd.DataFrame()
    master_summary = pd.concat(all_summary, ignore_index=True) if len(all_summary) > 0 else pd.DataFrame()

    out_dir = Path(artifact_root)
    out_dir.mkdir(parents=True, exist_ok=True)
    master_raw.to_csv(out_dir / 'first5_master_raw.csv', index=False)
    master_summary.to_csv(out_dir / 'first5_master_summary.csv', index=False)

    return master_raw, master_summary


def build_same_epsilon_same_budget_master_table(prototype_df, baseline_df=None):
    if pd is None:
        return None

    if prototype_df is None or len(prototype_df) == 0:
        return pd.DataFrame()

    proto_tbl = prototype_df.copy()
    proto_tbl['method'] = 'kl_prototype_dp'

    if baseline_df is None or len(baseline_df) == 0:
        cols = [
            'epsilon_total', 'method', 'mean_acc', 'std_acc', 'train_graphs_used',
            'val_graphs_used', 'mean_elapsed_sec', 'privacy_track', 'split_mode', 'seed_count'
        ]
        return proto_tbl[cols]

    base_tbl = baseline_df.copy()
    master = pd.concat([proto_tbl, base_tbl], ignore_index=True, sort=False)
    return master


if 'dataset' in globals() and 'data' in globals():
    LAUNCH_PLAN = launch_first_five_experiments(dataset_obj=dataset, data_obj=data, execute=False)

    print('First-5 launch plan prepared. Set EXECUTE_FIRST_FIVE = True to run.')
    if pd is not None:
        display(LAUNCH_PLAN)
    else:
        print(LAUNCH_PLAN)

    EXECUTE_FIRST_FIVE = True
    if EXECUTE_FIRST_FIVE:
        master_raw_df, master_summary_df = launch_first_five_experiments(
            dataset_obj=dataset,
            data_obj=data,
            execute=True,
            artifact_root='artifacts/paper_default_v1',
        )
        if pd is not None:
            display(master_summary_df)
else:
    print('Run the dataset setup cell first so dataset/data are available, then re-run this cell.')

In [ ]:

# Save the master summary dataframe from the 'First-5' launcher to a CSV
if 'master_summary_df' in globals() and master_summary_df is not None:
    output_filename = 'first5_final_results_summary.csv'
    master_summary_df.to_csv(output_filename, index=False)
    print(f'Successfully saved results to {output_filename}')
else:
    print('master_summary_df not found. Please ensure Cell 8 has finished executing.')
